# TFM WP-04 — Brazil 3M DI (YCSW0089)

Reproducible Bloomberg BQuant/BQL extraction of the monthly 3M point from the BM&F Pre x DI curve (`YCSW0089 Index`) for 2010-12-31 through 2025-12-31.

This series is intended for the WP-04 referee-only robustness checks. It does **not** replace the existing `fed_3m_forward` variable.

In [ ]:
import bql
import pandas as pd
from pathlib import Path

bq = bql.Service()

CURVE = "YCSW0089 Index"
TENOR = "3M"
START_DATE = "2010-12-31"
END_DATE = "2025-12-31"
OUTPUT_FILENAME = "brazil_3m_di_ycsw0089_monthly.csv" 

## Monthly observation dates

BQNT currently supports pandas month-end frequency `M` in this environment.

In [ ]:
dates = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="M"
)

print(f"Number of month-end observations: {len(dates)}")
print(dates.min(), "to", dates.max())

## Historical extraction

For each month-end, Bloomberg is asked for the historical 3M member of `YCSW0089 Index`. The returned member ticker is retained as an audit trail.

In [ ]:
records = []

for i, dt in enumerate(dates, start=1):
    d = dt.strftime("%Y-%m-%d")
    print(f"[{i:03d}/{len(dates):03d}] {d}", end=" | ")

    try:
        universe = bq.univ.curvemembers(
            symbols=CURVE,
            tenors=[TENOR],
            dates=d
        )

        items = {
            "Curve Rate": bq.data.curve_rate(dates=d),
            "Ticker": bq.data.id(),
        }

        req = bql.Request(universe, items)
        res = bq.execute(req)
        rate_df = res[0].df()

        if rate_df.empty:
            print("NO DATA")
            records.append({
                "obs_date": d,
                "curve": CURVE,
                "tenor": TENOR,
                "curve_rate": None,
                "member_ticker": None,
                "bloomberg_date": None,
                "status": "NO_DATA",
            })
            continue

        member_ticker = str(rate_df.index[0])
        curve_rate = rate_df["Curve Rate"].iloc[0] if "Curve Rate" in rate_df.columns else None
        bloomberg_date = rate_df["DATE"].iloc[0] if "DATE" in rate_df.columns else None

        records.append({
            "obs_date": d,
            "curve": CURVE,
            "tenor": TENOR,
            "curve_rate": curve_rate,
            "member_ticker": member_ticker,
            "bloomberg_date": bloomberg_date,
            "status": "OK",
        })

        print(f"OK | {member_ticker} | {curve_rate}")

    except Exception as exc:
        print(f"ERROR | {exc}")
        records.append({
            "obs_date": d,
            "curve": CURVE,
            "tenor": TENOR,
            "curve_rate": None,
            "member_ticker": None,
            "bloomberg_date": None,
            "status": f"ERROR: {exc}",
        })

## Build and validate the monthly series

In [ ]:
df = pd.DataFrame(records)

df["obs_date"] = pd.to_datetime(df["obs_date"], errors="coerce")
df["bloomberg_date"] = pd.to_datetime(df["bloomberg_date"], errors="coerce")
df["curve_rate"] = pd.to_numeric(df["curve_rate"], errors="coerce")
df = df.sort_values("obs_date").reset_index(drop=True)

expected_n = len(dates)
ok_n = int((df["status"] == "OK").sum())
missing_rates = int(df["curve_rate"].isna().sum())
duplicate_dates = int(df["obs_date"].duplicated().sum())
date_mismatches = int((
    df["bloomberg_date"].notna()
    & (df["obs_date"].dt.to_period("M") != df["bloomberg_date"].dt.to_period("M"))
).sum())

print("Expected observations      :", expected_n)
print("Downloaded observations    :", len(df))
print("Successful observations    :", ok_n)
print("Missing curve rates        :", missing_rates)
print("Duplicate observation dates:", duplicate_dates)
print("Month mismatches           :", date_mismatches)

display(df.head())
display(df.tail())
display(df["curve_rate"].describe())

## Save CSV in the BQuant project

Download the resulting CSV from the BQuant project and place it locally at:

`C:\Users\tsiqueira4\PycharmProjects\tmf_thiago_siqueira\datos_y_modelos\db\brazil_macroeconomic_data\brazil_3m_di_forward`

In [ ]:
output_path = Path(OUTPUT_FILENAME)
df.to_csv(output_path, index=False)
print("Saved:", output_path.resolve())